# S216 — Bellman-Ford Closure-to-Barrier Certificates (Colab Demo)

Reproduces the certificate-generation pipeline for the $\mathcal{C}^{\leftarrow}_{S202}(m, Q) \ge T$ barrier theorems.

Closure rule (S216): a state-cost pair $(v, d)$ is *relevant* iff
$$ d - (Q - j(v)) < T. $$

This is the **optimistic** cut required for a formal barrier when negative edges exist in the inverse cylinder graph.

## 0. Setup

In [ ]:
# If running on Colab, upload `s216_colab.py` or fetch from your fork.
import os
if not os.path.exists('s216_colab.py'):
    raise SystemExit('Upload s216_colab.py to this Colab session first.')

from s216_colab import S216Colab, run_colab
print('S216 engine loaded.')

## 1. Small cases (sanity)
Reproduces the (m, Q) cases from the paper. These complete in seconds.

In [ ]:
for (m, Q) in [(1, 3), (1, 5), (1, 8), (2, 6)]:
    cert = run_colab(m, Q, budget=200_000)
    print(f'm={m} Q={Q} → {cert["status"]:<24}'
          f' states={cert.get("states_seen", "?")} '
          f'verify={cert.get("_verify", {}).get("ok")}')

## 2. (m=1, Q=10) — large case (was budget-exceeded on local 200K)

On Colab with ~10M budget this completes. Use a checkpoint so you can resume if disconnected.

In [ ]:
cert = run_colab(m=1, Q=10, budget=5_000_000,
                 checkpoint='/content/s216_m1Q10.pkl')
print('Status:', cert['status'])
if cert['status'] == 'barrier_certificate':
    print('  states:', cert['states_seen'])
    print('  conclusion:', cert['conclusion'])
    print('  verify:', cert.get('_verify', {}).get('ok'))

## 3. Persist the certificate (JSON)
Download the certificate for offline lifting to a Lean module.

In [ ]:
import json
with open('/content/cert_m1_Q10_S216.json', 'w') as f:
    json.dump(cert, f, indent=2, ensure_ascii=False)
from google.colab import files
files.download('/content/cert_m1_Q10_S216.json')

## 4. Verify an external certificate (independent verifier)

Standard sanity check: load the JSON, instantiate the engine in *verify mode*, re-check the three closure invariants.

In [ ]:
import json
with open('/content/cert_m1_Q10_S216.json') as f:
    cert_loaded = json.load(f)
engine = S216Colab(cert_loaded['m'], cert_loaded['Q'], cert_loaded['T'])
verdict = engine.verify(cert_loaded)
print(json.dumps(verdict, indent=2))

## 5. Sweep `Q` for fixed `m=1`
Diagnostic: how does the table grow?

In [ ]:
rows = []
for Q in [3, 5, 8, 10]:
    cert = run_colab(m=1, Q=Q, budget=5_000_000)
    rows.append((Q, cert['states_seen'], cert['status']))
    print(f'Q={Q}: {cert["status"]:<24} states={cert["states_seen"]}')

print()
print('Q  | states  | status')
for Q, n, st in rows:
    print(f'{Q:>2} | {n:>7} | {st}')